# Notebook 14: Embedding Drift Detection

## Why This Matters
Your RAG pipeline is live. Retrieval recall was 0.87 at launch. Six months later it's 0.71 and nobody noticed because it degraded gradually. The corpus grew. User queries evolved. The embedding model was updated to a new version. Any of these can shift the geometry of your embedding space and silently break retrieval.

Embedding drift detection monitors these shifts statistically so you catch degradation before it affects users — this is the `drift-cam` foundation for image embeddings and the `drift` foundation for LLM applications.

## Structure
1. What causes embedding drift (narrative)
2. Statistical tests for distribution shift
3. Monitoring query embedding distributions
4. Monitoring corpus embedding distributions
5. Setting alert thresholds
6. Bridge to cross-modal retrieval

## What You'll Learn
- How KS test, MMD, and Jensen-Shannon divergence detect distributional shift
- How to build a query distribution monitor with a sliding window
- How to detect when new corpus documents are out-of-distribution
- How to set alert thresholds without too many false positives


## Background

### What causes embedding drift

**Query drift:** User behavior changes. Queries shift from short keyword-style to long conversational. A new product launch creates a burst of unfamiliar queries.

**Corpus drift:** New documents are added that cover topics not in the original corpus. The centroid of the corpus embedding distribution shifts.

**Model drift:** The embedding model is updated (e.g., from `all-MiniLM-L6-v2` to `all-mpnet-base-v2`). All embeddings are now in a different space — old index is invalid.

**Reference distribution shift:** Your held-out evaluation set is no longer representative of production.

### Statistical tests

**KS test (Kolmogorov-Smirnov):** Compares CDFs of two 1D distributions. Apply it to a low-dimensional projection (e.g., first PCA component) of embeddings. Fast, non-parametric, interpretable p-value. Works per-dimension — run it on the top k PCA components and aggregate.

**Maximum Mean Discrepancy (MMD):** Kernel-based two-sample test that operates in the original embedding space. More powerful than per-dimension KS but slower. The right choice for detecting subtle shifts in high-dimensional embeddings.

**Jensen-Shannon divergence:** Measures divergence between two probability distributions. Requires fitting a density model first — use kernel density estimation on a 2D projection. Good for visualization.

**Cosine similarity to reference centroid:** Fast and intuitive — compute the mean embedding of a reference window, then monitor how much new embeddings deviate from it. Not a statistical test but useful for real-time alerting.


In [ ]:
# %pip install -q sentence-transformers numpy scipy scikit-learn matplotlib torch

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import rbf_kernel
from sentence_transformers import SentenceTransformer

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
np.random.seed(42)

## 1. Simulating Reference and Drifted Distributions

In [ ]:
# Simulate reference query embeddings (at launch)
reference_queries = [
    "how does flash attention work", "what is the kv cache",
    "explain speculative decoding", "how does quantization reduce memory",
    "what is continuous batching", "how does paged attention work",
    "explain beam search", "what is temperature in sampling",
    "how does LoRA fine-tuning work", "what is RLHF",
    "how does tensor parallelism work", "explain pipeline parallelism",
    "what is the residual stream", "how does layernorm work",
    "what are the components of a transformer block",
] * 5

# Simulate drifted queries (6 months later — users ask about new topics)
drifted_queries = [
    "compare claude and gpt-4", "which model is best for coding",
    "how do I use the openai api", "what is function calling",
    "explain chain of thought prompting", "how does RAG improve accuracy",
    "what is prompt engineering", "how to reduce hallucinations",
    "explain agent frameworks like langchain", "what is a system prompt",
    "how do I fine-tune a model", "what is PEFT",
    "explain multimodal models", "how does vision work in gpt-4v",
    "what is context length and why does it matter",
] * 5

ref_embs  = model.encode(reference_queries,  show_progress_bar=False)
drift_embs = model.encode(drifted_queries, show_progress_bar=False)

print(f"Reference embeddings: {ref_embs.shape}")
print(f"Drifted embeddings:   {drift_embs.shape}")
print(f"Embedding dim: {ref_embs.shape[1]}")

## 2. KS Test on PCA Projections

In [ ]:
# Project to lower dims and run KS test per component
pca = PCA(n_components=10)
pca.fit(ref_embs)

ref_pca   = pca.transform(ref_embs)
drift_pca = pca.transform(drift_embs)

print("KS test per PCA component (reference vs drifted):")
print(f"{'Component':<12} {'KS stat':>9} {'p-value':>10}  {'Significant?':>14}")
print("-" * 50)

ks_stats, p_values = [], []
for i in range(10):
    stat, pval = stats.ks_2samp(ref_pca[:, i], drift_pca[:, i])
    ks_stats.append(stat); p_values.append(pval)
    sig = "YES ← DRIFT" if pval < 0.05 else "no"
    print(f"  PC{i+1:02d}       {stat:>9.4f} {pval:>10.4f}  {sig:>14}")

# Aggregate: Bonferroni-corrected
n_tests = 10
bonferroni_threshold = 0.05 / n_tests
n_significant = sum(p < bonferroni_threshold for p in p_values)
print(f"\nSignificant components (Bonferroni α={bonferroni_threshold:.3f}): {n_significant}/{n_tests}")
print(f"Drift detected: {'YES' if n_significant > 0 else 'NO'}")

## 3. Maximum Mean Discrepancy

In [ ]:
def mmd_rbf(X: np.ndarray, Y: np.ndarray, gamma: float = None) -> float:
    """Maximum Mean Discrepancy with RBF kernel.
    
    MMD² = E[k(x,x')] - 2E[k(x,y)] + E[k(y,y')]
    where k is the RBF kernel. MMD=0 if distributions are identical.
    """
    if gamma is None:
        # Median heuristic for bandwidth
        all_data = np.vstack([X, Y])
        pairwise = np.sum((all_data[:, None] - all_data[None, :])**2, axis=-1)
        gamma = 1.0 / np.median(pairwise[pairwise > 0])

    Kxx = rbf_kernel(X, X, gamma=gamma)
    Kyy = rbf_kernel(Y, Y, gamma=gamma)
    Kxy = rbf_kernel(X, Y, gamma=gamma)

    m, n = len(X), len(Y)
    mmd2 = (Kxx.sum() - Kxx.diagonal().sum()) / (m * (m-1))          + (Kyy.sum() - Kyy.diagonal().sum()) / (n * (n-1))          - 2 * Kxy.mean()
    return float(max(0, mmd2))


# Use PCA-reduced embeddings for speed (MMD is O(N²))
X_ref   = pca.transform(ref_embs)
X_drift = pca.transform(drift_embs)

# Baseline: MMD between two splits of the reference set (should be ~0)
mid = len(X_ref) // 2
mmd_baseline = mmd_rbf(X_ref[:mid], X_ref[mid:])
mmd_drift    = mmd_rbf(X_ref, X_drift)

print(f"MMD² (reference vs reference split): {mmd_baseline:.6f}  (baseline, ~0 expected)")
print(f"MMD² (reference vs drifted):         {mmd_drift:.6f}")
print(f"Ratio: {mmd_drift / (mmd_baseline + 1e-8):.1f}×")
print()
print(f"Alert threshold: if MMD > 10× baseline → investigate")
alert = mmd_drift > 10 * mmd_baseline
print(f"Drift alert: {'TRIGGERED' if alert else 'clear'}")

## 4. Sliding Window Monitor

In [ ]:
def sliding_window_drift_monitor(reference_embs: np.ndarray,
                                  stream_embs: np.ndarray,
                                  window_size: int = 20,
                                  step: int = 5,
                                  pca_components: int = 5):
    """Monitor embedding drift over a stream with a sliding window."""
    pca_mon = PCA(n_components=pca_components).fit(reference_embs)
    ref_proj = pca_mon.transform(reference_embs)

    timestamps, ks_scores = [], []

    for start in range(0, len(stream_embs) - window_size + 1, step):
        window = stream_embs[start:start + window_size]
        win_proj = pca_mon.transform(window)

        # KS stat across all PCA components — use max as aggregate
        max_ks = max(stats.ks_2samp(ref_proj[:, i], win_proj[:, i]).statistic
                     for i in range(pca_components))
        timestamps.append(start + window_size)
        ks_scores.append(max_ks)

    return timestamps, ks_scores


# Simulate a stream: first half is reference-like, second half drifts
stream = np.vstack([
    ref_embs[np.random.choice(len(ref_embs), 30)],    # first 30: normal
    drift_embs[np.random.choice(len(drift_embs), 45)], # last 45: drifted
])

timestamps, ks_scores = sliding_window_drift_monitor(ref_embs, stream, window_size=15, step=3)

threshold = 0.3  # tune based on false positive tolerance
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(timestamps, ks_scores, color='royalblue', linewidth=2)
ax.axhline(threshold, color='red', linestyle='--', label=f'Alert threshold ({threshold})')
ax.axvline(30, color='orange', linestyle=':', label='Drift starts (t=30)')
ax.set_xlabel("Stream position"); ax.set_ylabel("Max KS statistic")
ax.set_title("Embedding Drift Monitor — Sliding Window")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Setting Alert Thresholds

```
Conservative (few false positives):  KS > 0.4  or  MMD > 20× baseline
Balanced:                            KS > 0.3  or  MMD > 10× baseline
Sensitive (catch early drift):       KS > 0.2  or  MMD > 5× baseline
```

**Calibrate on your data:**
1. Split reference data in half 100 times, compute KS/MMD distribution → this is your null
2. Set threshold at 95th or 99th percentile of the null distribution
3. Run on known-drifted data to check power (true positive rate)

**What to do when drift is detected:**
1. Check query logs — are users asking new kinds of questions?
2. Check corpus additions — were many documents added recently?
3. Re-evaluate retrieval recall@k on recent queries
4. If recall degraded: consider fine-tuning or reindexing with a better model


## 6. Bridge to Cross-Modal Retrieval

Drift detection works on any embedding space. Notebook 15 applies the same ideas to CLIP embeddings for image streams — detecting when production images have drifted from the distribution the retrieval system was calibrated on. This is the `drift-cam` prototype.

But first: Notebook 15 covers cross-modal retrieval itself — how to search images with text queries and vice versa using CLIP, and how to evaluate recall@k across modalities.


## Summary

| Method | When to use | Sensitivity | Speed |
|--------|-------------|-------------|-------|
| KS test on PCA dims | Routine monitoring, interpretable | Moderate | Fast |
| MMD | Subtle distributional shifts | High | O(N²) |
| Centroid distance | Real-time alerting | Low (catches large shifts) | O(1) |
| Sliding window | Time-series monitoring | Configurable | Moderate |

**Next:** Notebook 15 — Cross-Modal Retrieval. CLIP-based image↔text search pipelines and recall@k evaluation across modalities.
